# 08 · Evaluación comparativa de los modelos de keypoints sobre test propio

**Objetivo:** evaluar los cinco modelos de keypoints (A base, B, C, D, E) sobre un conjunto de test propio de 50 frames de Bundesliga, revisado manualmente, que ningún modelo ha visto durante el entrenamiento.

**Contexto:** hasta ahora los modelos se compararon sobre validación, parte de los datos de entrenamiento. Esta evaluación sobre test independiente proporciona la comparación rigurosa y definitiva, respetando la metodología aplicada también en el detector.

**Entrada:** anotaciones revisadas (COCO) e imágenes del test, y los modelos en `models/`.

**Salida:** dataset de test en formato YOLO-pose y tabla comparativa de los 5 modelos.

In [1]:
import json
import shutil
from pathlib import Path

REPO_ROOT = Path("../..").resolve()

# JSON revisado exportado de CVAT
COCO_JSON = Path(r"C:\Users\sgghu\OneDrive\Desktop\UNIVERSIDAD\TRABAJAJOFINGRADO\annotations\person_keypoints_default.json")

# Imágenes del test (las que subiste a CVAT)
IMAGES_SRC = REPO_ROOT / "data" / "cvat_import_test" / "images"

# Salida: carpeta test dentro del dataset de bundesliga
TEST_DIR = REPO_ROOT / "data" / "bundesliga_keypoints" / "test"

N_KP = 24

print(f"JSON existe: {COCO_JSON.exists()}")
print(f"Imágenes existen: {IMAGES_SRC.exists()}")

JSON existe: True
Imágenes existen: True


In [2]:
with open(COCO_JSON, encoding="utf-8") as f:
    coco = json.load(f)

print(f"Imágenes en JSON: {len(coco['images'])}")
print(f"Anotaciones en JSON: {len(coco['annotations'])}")

img_info = {img["id"]: img for img in coco["images"]}

# Crear estructura test
(TEST_DIR / "images").mkdir(parents=True, exist_ok=True)
(TEST_DIR / "labels").mkdir(parents=True, exist_ok=True)

ann_por_img = {}
for ann in coco["annotations"]:
    ann_por_img.setdefault(ann["image_id"], []).append(ann)

procesados = 0
no_encontradas = 0

for img_id, anns in ann_por_img.items():
    info = img_info[img_id]
    w, h = info["width"], info["height"]
    fname_base = Path(info["file_name"]).name
    src_img = IMAGES_SRC / fname_base
    if not src_img.exists():
        no_encontradas += 1
        continue

    ann = anns[0]
    kps = ann["keypoints"]
    partes = ["0", "0.5", "0.5", "1.0", "1.0"]
    for i in range(N_KP):
        x, y, v = kps[i*3], kps[i*3+1], kps[i*3+2]
        if v >= 2:
            partes.extend([f"{x/w:.6f}", f"{y/h:.6f}", "2"])
        else:
            partes.extend(["0", "0", "0"])

    shutil.copy2(src_img, TEST_DIR / "images" / fname_base)
    with open(TEST_DIR / "labels" / (Path(fname_base).stem + ".txt"), "w") as f:
        f.write(" ".join(partes) + "\n")
    procesados += 1

print(f"Test procesado: {procesados} imágenes")
print(f"No encontradas: {no_encontradas}")

Imágenes en JSON: 50
Anotaciones en JSON: 50
Test procesado: 50 imágenes
No encontradas: 0


## Evaluación de los modelos sobre el test

Se crea un data.yaml que apunta al conjunto de test y se evalúa cada modelo con `split='test'`. El modelo A (base SoccerNet) y los cuatro fine-tunings se comparan sobre las mismas 50 imágenes.

In [3]:
# data.yaml apuntando al test
yaml_content = f"""train: {REPO_ROOT}/data/bundesliga_keypoints/train/images
val: {REPO_ROOT}/data/bundesliga_keypoints/valid/images
test: {TEST_DIR}/images

kpt_shape: [24, 3]
nc: 1
names: ['field']
"""
yaml_path = REPO_ROOT / "data" / "bundesliga_keypoints" / "data_test.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("data_test.yaml creado")

data_test.yaml creado


In [4]:
from ultralytics import YOLO

MODELS_DIR = REPO_ROOT / "models"

modelos = {
    "A (base SoccerNet)":      "keypoints_base.pt",
    "B (FT completo)":         "keypoints_ft_B.pt",
    "C (FT backbone)":         "keypoints_ft_C.pt",
    "D (FT cabeza)":           "keypoints_ft_D.pt",
    "E (FT backbone 1280)":    "keypoints_ft_E.pt",
}

resultados = {}

for nombre, archivo in modelos.items():
    ruta = MODELS_DIR / archivo
    if not ruta.exists():
        print(f"  AVISO: no existe {archivo}, se omite")
        continue
    print(f"Evaluando {nombre}...")
    model = YOLO(str(ruta))
    # E se evaluó a 1280; el resto a 640
    imgsz = 1280 if "1280" in archivo else 640
    metrics = model.val(data=str(yaml_path), split="test", imgsz=imgsz, workers=0, verbose=False)
    resultados[nombre] = {
        "pose_mAP50": metrics.pose.map50,
        "pose_mAP50_95": metrics.pose.map,
    }
    print(f"  Pose mAP50: {metrics.pose.map50:.3f}")

print("\nEvaluación completada")

Evaluando A (base SoccerNet)...
Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLOv8s-pose summary (fused): 82 layers, 11,862,747 parameters, 0 gradients, 31.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 529.877.3 MB/s, size: 492.9 KB)
val: Scanning C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\bundesliga_keypoints\test\labels... 50 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 50/50 1.2Kit/s 0.0s
val: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\bundesliga_keypoints\test\images\A1606b0e6_0_(37)_A1606b0e6_0_(37)_A1606b0e6_0_(37)_frame00732.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.011296]
val: New cache created: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\bundesliga_keypoints\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.9it/s 2.2s1.0s
   

In [5]:
print("COMPARATIVA SOBRE TEST PROPIO (50 frames Bundesliga)\n")
print(f"{'Modelo':<28} {'Pose mAP50':>12} {'Pose mAP50-95':>14}")
print("-" * 56)
for nombre, r in resultados.items():
    print(f"{nombre:<28} {r['pose_mAP50']:>12.3f} {r['pose_mAP50_95']:>14.3f}")
print("-" * 56)

# Identificar el mejor por mAP50-95
mejor = max(resultados.items(), key=lambda x: x[1]["pose_mAP50_95"])
print(f"\nMejor modelo (por mAP50-95): {mejor[0]}")

COMPARATIVA SOBRE TEST PROPIO (50 frames Bundesliga)

Modelo                         Pose mAP50  Pose mAP50-95
--------------------------------------------------------
A (base SoccerNet)                  0.806          0.458
B (FT completo)                     0.801          0.431
C (FT backbone)                     0.690          0.450
D (FT cabeza)                       0.652          0.438
E (FT backbone 1280)                0.741          0.365
--------------------------------------------------------

Mejor modelo (por mAP50-95): A (base SoccerNet)


In [ ]:
# ---- Añadir el modelo SoccerNet-completo a la comparativa sobre test propio ----
from ultralytics import YOLO
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
MODELO_NUEVO = REPO_ROOT / "models" / "keypoints_ft_soccernet.pt"
DATA_TEST = REPO_ROOT / "data" / "bundesliga_keypoints" / "data.yaml"  # test propio (50 frames)

print(f"Modelo nuevo existe: {MODELO_NUEVO.exists()}")

model = YOLO(str(MODELO_NUEVO))
metrics = model.val(data=str(DATA_TEST), split="test", imgsz=640, device=0, verbose=False)

map50 = metrics.pose.map50
map5095 = metrics.pose.map
print(f"\n=== keypoints_ft_soccernet (entrenado con 10.696 frames SoccerNet) ===")
print(f"  Pose mAP50:    {map50:.3f}")
print(f"  Pose mAP50-95: {map5095:.3f}")
print(f"\nComparar contra tu tabla previa:")
print(f"  A (base SoccerNet):  mAP50=0.806  mAP50-95=0.458")
print(f"  B (FT 96 frames):    mAP50=0.801  mAP50-95=0.431")
print(f"\n  -> Si el nuevo supera 0.806 en mAP50, el dataset grande mejoró el modelo.")
print(f"  -> AVISO: este test son 50 frames de Bundesliga. La prueba decisiva es el")
print(f"     nº de frames con homografía válida en el notebook 03 (fase 4), no este mAP.")